# Test III: Quantum Machine Learning — Hybrid QCNN with CUDA-Q

**Task:** Build a quantum machine learning model for 3-class gravitational lensing classification.

**Approach:** Hybrid Quanvolutional CNN — replace the first classical convolution layer with a **Quantum Convolution Layer**, where a parameterized quantum circuit (PQC) acts as a trainable convolutional filter.

**Framework:** NVIDIA CUDA-Q for quantum circuit simulation (GPU-accelerated), PyTorch for classical layers.

## Architecture

```
Input (1, 150, 150)
    │
    ├── Downsample to (1, 28, 28) via AvgPool
    │
    ├── [Quantum Conv Layer]  2×2 kernel, stride=2 → 4 qubits PQC → 4 output channels
    │       → output: (4, 14, 14)
    │
    ├── [Classical Conv2d + BN + ReLU]  4 → 16 channels
    ├── [Classical Conv2d + BN + ReLU]  16 → 32 channels
    ├── [AdaptiveAvgPool → Flatten]
    └── [FC → 3 classes]
```

### Quantum Convolution Design

- **Kernel size:** 2×2 = 4 pixels → 4 qubits
- **Data encoding:** Angle encoding — each pixel value scaled to [0, π] as Ry rotation
- **Variational circuit:** Multiple layers of Ry/Rz rotations + ring CNOT entanglement
- **Readout:** ⟨Z_i⟩ expectation values for each qubit → 4 output channels
- **Gradient:** Parameter-shift rule integrated into PyTorch autograd
- **Stride:** 2 (non-overlapping patches for efficiency)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Function
import cudaq
from cudaq import spin
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
import time
import warnings
warnings.filterwarnings('ignore')

# Use GPU simulator if available
try:
    cudaq.set_target('nvidia')
    print('Using CUDA-Q nvidia (GPU) target')
except:
    print('Using CUDA-Q default (CPU) target')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch device: {device}')

## 1. Dataset

In [ ]:
CLASS_NAMES = ['no', 'sphere', 'vort']
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

class LensingDataset(Dataset):
    def __init__(self, root_dir, augment=False, downsample_size=28):
        self.samples = []
        self.labels = []
        self.augment = augment
        self.pool = nn.AvgPool2d(kernel_size=150 // downsample_size, stride=150 // downsample_size)
        self.downsample_size = downsample_size
        for class_name in CLASS_NAMES:
            class_dir = os.path.join(root_dir, class_name)
            for fname in sorted(os.listdir(class_dir)):
                if fname.endswith('.npy'):
                    self.samples.append(os.path.join(class_dir, fname))
                    self.labels.append(CLASS_TO_IDX[class_name])
        self.labels = np.array(self.labels)
        print(f'Loaded {len(self.samples)} samples from {root_dir}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img = np.load(self.samples[idx]).astype(np.float32)  # (1, 150, 150)
        label = self.labels[idx]
        if self.augment:
            if np.random.rand() > 0.5:
                img = img[:, :, ::-1].copy()
            if np.random.rand() > 0.5:
                img = img[:, ::-1, :].copy()
            k = np.random.randint(0, 4)
            img = np.rot90(img, k, axes=(1, 2)).copy()
        img_t = torch.from_numpy(img)
        img_t = self.pool(img_t)  # (1, 28, 28) after downsampling 150/5=30... use adaptive
        img_t = nn.functional.adaptive_avg_pool2d(img_t, self.downsample_size)
        return img_t, label

In [ ]:
DOWNSAMPLE = 28  # 150×150 → 28×28 (manageable for quantum conv)

train_dataset = LensingDataset('dataset/dataset/train', augment=True, downsample_size=DOWNSAMPLE)
val_dataset = LensingDataset('dataset/dataset/val', augment=False, downsample_size=DOWNSAMPLE)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

# Verify shape
img, label = train_dataset[0]
print(f'Image shape after downsample: {img.shape}')  # (1, 28, 28)

## 2. Quantum Convolution Layer

The quantum convolutional filter works as follows:
1. Extract a 2×2 patch from the image (4 pixel values)
2. Encode each pixel as a Ry rotation on a qubit (angle encoding: θ = pixel × π)
3. Apply variational layers: trainable Ry/Rz rotations + CNOT entanglement
4. Measure ⟨Z⟩ on each qubit → 4 output values per patch
5. Slide across the image with stride 2 to produce output feature maps

In [ ]:
N_QUBITS = 4       # 2×2 kernel
N_LAYERS = 2        # Variational layers
# Each layer: N_QUBITS Ry + N_QUBITS Rz = 2 * N_QUBITS params per layer
N_PARAMS = N_LAYERS * 2 * N_QUBITS  # 16 parameters

print(f'Quantum circuit: {N_QUBITS} qubits, {N_LAYERS} layers, {N_PARAMS} trainable params')

In [ ]:
# Define the quantum convolutional filter kernel
@cudaq.kernel
def qconv_kernel(data: list[float], params: list[float],
                 n_qubits: int, n_layers: int):
    """Quantum convolutional filter.
    
    Args:
        data: 4 pixel values from a 2×2 patch (angle encoded)
        params: trainable variational parameters
        n_qubits: number of qubits (4 for 2×2 kernel)
        n_layers: number of variational layers
    """
    q = cudaq.qvector(n_qubits)
    
    for layer in range(n_layers):
        base = layer * 2 * n_qubits
        
        # Data encoding + trainable rotation (data re-uploading)
        for i in range(n_qubits):
            ry(data[i] * 3.14159265 + params[base + i], q[i])
        
        # Entanglement: ring of CNOTs
        for i in range(n_qubits - 1):
            x.ctrl(q[i], q[i + 1])
        x.ctrl(q[n_qubits - 1], q[0])
        
        # Additional trainable Rz rotation
        for i in range(n_qubits):
            rz(params[base + n_qubits + i], q[i])
    
    # NO measurement — cudaq.observe handles this

In [ ]:
# Define observables: Z on each qubit → 4 output channels
observables = [spin.z(i) for i in range(N_QUBITS)]

def run_qconv_single(data_patch, params):
    """Run quantum conv on a single 2×2 patch, return 4 expectation values."""
    results = []
    for obs in observables:
        result = cudaq.observe(qconv_kernel, obs,
                               data_patch, params, N_QUBITS, N_LAYERS)
        results.append(result.expectation())
    return results

# Quick test
test_data = [0.5, 0.3, 0.7, 0.1]
test_params = np.random.randn(N_PARAMS).tolist()
out = run_qconv_single(test_data, test_params)
print(f'Quantum conv test output: {out}')

In [ ]:
class QuantumConvFunction(Function):
    """PyTorch autograd Function wrapping the quantum convolutional filter.
    Uses parameter-shift rule for gradient computation."""
    
    @staticmethod
    def forward(ctx, input_img, params, kernel_size, stride):
        """
        Args:
            input_img: (B, 1, H, W) tensor
            params: (N_PARAMS,) tensor — quantum circuit parameters
            kernel_size: int (2)
            stride: int (2)
        Returns:
            output: (B, N_QUBITS, H_out, W_out) tensor
        """
        B, C, H, W = input_img.shape
        H_out = (H - kernel_size) // stride + 1
        W_out = (W - kernel_size) // stride + 1
        
        params_list = params.detach().cpu().numpy().tolist()
        input_np = input_img.detach().cpu().numpy()
        
        output = np.zeros((B, N_QUBITS, H_out, W_out))
        
        for b in range(B):
            for i in range(H_out):
                for j in range(W_out):
                    # Extract 2×2 patch
                    r, c = i * stride, j * stride
                    patch = input_np[b, 0, r:r+kernel_size, c:c+kernel_size].flatten().tolist()
                    # Run quantum circuit
                    exp_vals = run_qconv_single(patch, params_list)
                    for q_idx in range(N_QUBITS):
                        output[b, q_idx, i, j] = exp_vals[q_idx]
        
        output_tensor = torch.tensor(output, dtype=torch.float32)
        ctx.save_for_backward(input_img, params)
        ctx.kernel_size = kernel_size
        ctx.stride = stride
        return output_tensor
    
    @staticmethod
    def backward(ctx, grad_output):
        """Parameter-shift rule: df/dθ_k = [f(θ_k + π/2) - f(θ_k - π/2)] / 2"""
        input_img, params = ctx.saved_tensors
        kernel_size = ctx.kernel_size
        stride = ctx.stride
        shift = np.pi / 2
        
        B, C, H, W = input_img.shape
        H_out = (H - kernel_size) // stride + 1
        W_out = (W - kernel_size) // stride + 1
        
        input_np = input_img.detach().cpu().numpy()
        params_np = params.detach().cpu().numpy()
        grad_out_np = grad_output.detach().cpu().numpy()
        
        grad_params = np.zeros_like(params_np)
        
        for k in range(len(params_np)):
            # Shifted parameters
            params_plus = params_np.copy()
            params_plus[k] += shift
            params_minus = params_np.copy()
            params_minus[k] -= shift
            
            for b in range(B):
                for i in range(H_out):
                    for j in range(W_out):
                        r, c = i * stride, j * stride
                        patch = input_np[b, 0, r:r+kernel_size, c:c+kernel_size].flatten().tolist()
                        
                        exp_plus = run_qconv_single(patch, params_plus.tolist())
                        exp_minus = run_qconv_single(patch, params_minus.tolist())
                        
                        for q_idx in range(N_QUBITS):
                            grad_shift = (exp_plus[q_idx] - exp_minus[q_idx]) / 2.0
                            grad_params[k] += grad_out_np[b, q_idx, i, j] * grad_shift
        
        return None, torch.tensor(grad_params, dtype=torch.float32), None, None

In [ ]:
class QuantumConv2d(nn.Module):
    """Quantum Convolutional Layer — drop-in replacement for nn.Conv2d.
    
    Uses a parameterized quantum circuit as the convolutional filter.
    Input: (B, 1, H, W)
    Output: (B, n_qubits, H_out, W_out)
    """
    def __init__(self, kernel_size=2, stride=2, n_params=N_PARAMS):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        # Trainable quantum parameters
        self.params = nn.Parameter(torch.randn(n_params) * 0.1)
    
    def forward(self, x):
        return QuantumConvFunction.apply(x, self.params, self.kernel_size, self.stride)

## 3. Hybrid QCNN Model

The first convolution is quantum, rest is classical.

In [ ]:
class HybridQCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        # Quantum convolution: (1, 28, 28) → (4, 14, 14)
        self.qconv = QuantumConv2d(kernel_size=2, stride=2)
        
        # Classical layers after quantum conv
        self.classical = nn.Sequential(
            nn.BatchNorm2d(N_QUBITS),
            nn.ReLU(),
            
            # Conv block 1: (4, 14, 14) → (16, 7, 7)
            nn.Conv2d(N_QUBITS, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Conv block 2: (16, 7, 7) → (32, 3, 3)
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            # Conv block 3: (32, 3, 3) → (64, 1, 1)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        x = self.qconv(x)       # Quantum conv — returns CPU tensor
        x = x.to(next(self.classical.parameters()).device)
        x = self.classical(x)
        x = self.classifier(x)
        return x

model = HybridQCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
q_params = model.qconv.params.numel()
c_params = total_params - q_params
print(f'Total params: {total_params:,} (Quantum: {q_params}, Classical: {c_params:,})')

## 4. Training

**Note:** The quantum convolution layer is significantly slower than classical conv due to circuit simulation. We use a smaller dataset subset for initial training and validate on the full val set.

In [ ]:
# For faster iteration, optionally subsample training data
# Remove this cell or set TRAIN_SUBSET=None to use full dataset
TRAIN_SUBSET = 3000  # Use 1000 per class for faster training; set to None for full

if TRAIN_SUBSET is not None:
    from torch.utils.data import Subset
    indices = []
    per_class = TRAIN_SUBSET // 3
    for c in range(3):
        class_indices = np.where(train_dataset.labels == c)[0]
        chosen = np.random.choice(class_indices, per_class, replace=False)
        indices.extend(chosen.tolist())
    np.random.shuffle(indices)
    train_subset = Subset(train_dataset, indices)
    train_loader_q = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=2)
    print(f'Training on {len(train_subset)} samples (subset)')
else:
    train_loader_q = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
    print(f'Training on full dataset ({len(train_dataset)} samples)')

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

NUM_EPOCHS = 20
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for batch_idx, (imgs, labels) in enumerate(train_loader_q):
        # Quantum conv runs on CPU via cudaq, rest on GPU
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)  # qconv takes CPU input internally
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
        
        if (batch_idx + 1) % 20 == 0:
            print(f'  Batch {batch_idx+1}/{len(train_loader_q)} | '
                  f'Loss: {loss.item():.4f} | Acc: {correct/total:.4f}', flush=True)
    
    train_loss = running_loss / total
    train_acc = correct / total

    # Validate
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            labels = labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += imgs.size(0)
    val_loss = running_loss / total
    val_acc = correct / total

    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - t0
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_qcnn.pt')

    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} ({elapsed:.0f}s) | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_title('Accuracy')
plt.tight_layout()
plt.show()

## 5. Evaluation — ROC Curve & AUC

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_qcnn.pt'))
model.eval()

all_probs = []
all_labels = []
with torch.no_grad():
    for imgs, labels in val_loader:
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_labels_bin = label_binarize(all_labels, classes=[0, 1, 2])

In [ ]:
# Plot ROC curves (one-vs-rest)
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e41a1c', '#377eb8', '#4daf4a']

for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=colors[i], lw=2, label=f'{class_name} (AUC = {roc_auc:.4f})')

macro_auc = roc_auc_score(all_labels_bin, all_probs, multi_class='ovr', average='macro')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — Hybrid QCNN with CUDA-Q (Macro AUC = {macro_auc:.4f})')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Macro AUC: {macro_auc:.4f}')

## 6. Discussion

### Strategy & Design Choices

**Data Encoding — Angle Encoding with Data Re-uploading:**
- Each pixel in the 2×2 patch is encoded as an Ry rotation: θ = pixel_value × π
- Data re-uploading (encoding repeated in each variational layer) increases the expressivity of the quantum model beyond what a single encoding pass can achieve

**Circuit Architecture — Variational Ansatz:**
- 4 qubits (matching the 2×2 kernel), 2 variational layers
- Each layer: parameterized Ry (with data) → ring CNOT entanglement → parameterized Rz
- The ring topology ensures all qubits are entangled, creating correlations between spatial pixels

**Readout — Pauli-Z Expectation Values:**
- Each qubit's ⟨Z⟩ produces one output channel (4 channels total)
- This is analogous to a classical Conv2d with 4 output filters
- Using cudaq.observe() gives analytic (noiseless) expectation values

**Hybrid Training:**
- Quantum parameters: gradient via parameter-shift rule
- Classical parameters: standard backpropagation
- Both optimized jointly with Adam

**Downsampling:**
- Images reduced from 150×150 to 28×28 to make quantum convolution tractable
- The quantum conv (stride 2) then produces 14×14 feature maps

**Why CUDA-Q:**
- GPU-accelerated state vector simulation via the `nvidia` target
- Significantly faster than CPU-based frameworks (Qiskit Aer, PennyLane default)
- Native support for `cudaq.observe()` with analytic gradients